# Dataset generation

| Var             | Range       | Unit | Description                             |
|-----------------|-------------|------|-----------------------------------------|
| $\eta_{PV}$     | 5-20        |      | panel efficiency                        |
| $\eta_{UGC}$    | 80-95       |      | PMU efficiency                          |
| $\eta_{B}$      | 80-90       |      | battery efficiency                      |
| $\eta_{STC}$    | 19-22       | %    | nominal efficiency of energy conversion |
| $A_{\text{PV}}$ |             | m²   | photovoltaic panel surface              |
| $P_{PV}$        |             | W    | panel power output                      |
| G(h)            |             | W/m² | hourly solar irradiance                 |
| Pn              |             | W    | power demand                            |
| Ppmu            |             | W    | PMU power                               |
| Pb              |             | W    | battery power                           |
| $\gamma$        | -0.29, -0.5 | %/ºC | temperature coefficient                 |

In [1]:
import itertools
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## Panel generation

In [ ]:
# load panels from panels.csv
panels_raw = pd.read_csv('panel_areas.csv')
panels_raw['Apv(m2)'] = (panels_raw['width_cm'].astype(float) / 100) * (panels_raw['height_cm'].astype(float) / 100)
Apv = sorted(panels_raw['Apv(m2)'].unique())
print(f"Loaded {len(panels_raw):,} panels, unique areas: {len(Apv)}")
display(panels_raw.head(10))

In [ ]:
# generate panels

eta_stc = [x / 10 for x in range(190, 221, 5)]
panels = list(itertools.product(Apv, eta_stc))
panels_df = pd.DataFrame(panels, columns=['Apv(m2)', 'eta_stc(%)'])
panels_df['Panel_ID'] = range(0, len(panels_df))
panels_df = panels_df[['Panel_ID', 'Apv(m2)', 'eta_stc(%)']]

print(f"# possible panels: {len(Apv)} × {len(eta_stc)} = {len(panels_df):,}")
print("First 10:")
display(panels_df.head(10))


## Irradiance data (Gh_data)

Load irradiance data from CSV file with hourly data for the whole year.

In [ ]:
# Column name constants (use these everywhere to refer to CSV columns)
COL_RAW_DATETIME = 'Date-hour'  # original datetime
COL_MONTH = 'Month'
COL_DAY = 'Day'
COL_HOUR = 'Hour'
COL_G = 'G(h)'
COL_TEMP = 'Temperature'

In [ ]:
# Columns: Month, Day, Hour, G(h), Temperature
Gh_data = pd.read_csv('raw-data/CR.csv', usecols=range(6))

# Rename columns to standard constants
# CSV header has: Date-hour, Month, Day, Hour, G(h), Temperature
Gh_data.columns = [COL_RAW_DATETIME, COL_MONTH, COL_DAY, COL_HOUR, COL_G, COL_TEMP]

print("📊 IRRADIANCE Gh_data")
print(f"Total records: {len(Gh_data):,}")
print(f"Available columns: {list(Gh_data.columns)}")
print(f"Dataset shape: {Gh_data.shape}")
print(f"Months: {Gh_data[COL_MONTH].min()} - {Gh_data[COL_MONTH].max()}")
print(f"Days: {Gh_data[COL_DAY].min()} - {Gh_data[COL_DAY].max()}")

# Show basic statistics for G(h)
gh_column = COL_G  # use constant
print(f"\n🌞 IRRADIANCE STATISTICS {gh_column}:")
print(f"Min: {Gh_data[gh_column].min():.2f} W/m²")
print(f"Max: {Gh_data[gh_column].max():.2f} W/m²")
print(f"Mean: {Gh_data[gh_column].mean():.2f} W/m²")
print(f"Records with {gh_column} > 0: {(Gh_data[gh_column] > 0).sum():,}")

print(f"\n🔍 FIRST 5 ROWS:")
display(Gh_data.head())

print(f"\n📊 COLUMN INFO:")
display(Gh_data.info())


## Power calculation

In [ ]:
def panel_power(Gh, Apv, nstc):
    """
    Calculate panel power output considering temperature effects.

    Parameters:
    - Gh: Irradiance (W/m²)
    - Apv: Panel area (m²)
    - nstc: Panel efficiency at standard test conditions (decimal)

    Returns:
    - Power output (W)
    """

    return Gh * Apv * nstc

def panel_power_with_temp(Gh, Apv, nstc, Tmod, temp_coeff=-0.004):
    """
    Calculate panel power output considering temperature effects.

    Parameters:
    - Gh: Irradiance (W/m²)
    - Apv: Panel area (m²)
    - nstc: Panel efficiency at standard test conditions (decimal)
    - Tmod: panel temperature (°C)
    - temp_coeff: Temperature coefficient (default -0.004 per °C)

    Returns:
    - Power output (W)
    """

    # Adjust efficiency based on temperature
    nPV = nstc * (1 + temp_coeff * (Tmod - 25))  # equation (3)
    return Gh * Apv * nPV  # equation (2)

In [ ]:
def get_irradiance(day, hour):
    if not (1 <= day <= 31 and 0 <= hour <= 23):
        return None

    row = Gh_data[(Gh_data[COL_DAY] == day) & (Gh_data[COL_HOUR] == hour)]
    if row.empty:
        return None

    return row.iloc[0][COL_G]

In [ ]:
# Calculate the power of each panel for each irradiance value

panel_powers = []
for idx, panel in panels_df.iterrows():
    Apv = panel['Apv(m2)']
    nstc = panel['eta_stc(%)'] / 100  # convert to fraction

    powers = []
    for Gh, Tmod in zip(Gh_data[COL_G], Gh_data[COL_TEMP]):
        powers.append(panel_power_with_temp(Gh, Apv, nstc, Tmod))
    panel_powers.append(powers)

panel_powers = pd.DataFrame(panel_powers)
panel_powers.index = panels_df['Panel_ID']
panel_powers.columns = [f"Gh_{i}" for i in range(len(panel_powers.columns))]

print("Power calculated for all panels and all irradiance values.")
display(panel_powers.head())

In [ ]:
# Calculate the total energy generated by each panel throughout the year (Wh)
# The energy per hour is equal to the power per hour (Wh)
total_panel_energy = panel_powers.sum(axis=1)
panels_with_energy = panels_df.copy()
panels_with_energy['Total_energy_Wh'] = total_panel_energy
print("Total energy generated by each panel throughout the year (Wh):")
display(panels_with_energy[['Panel_ID', 'Apv(m2)', 'eta_stc(%)', 'Total_energy_Wh']].head())

In [ ]:
# Calculate daily energy generated by each panel (Wh/day)
num_days = len(panel_powers.columns) // 24
day_labels = [f"day_{(i//24+1):03d}" for i in range(len(panel_powers.columns))]
panel_daily_energy = panel_powers.copy()
panel_daily_energy.columns = day_labels
panel_daily_energy = panel_daily_energy.T.groupby(level=0).sum().T
panel_daily_energy.index = panels_df['Panel_ID']
print("Daily energy generated by each panel (Wh/day):")
display(panel_daily_energy.head())

In [ ]:
# Plot daily energy for the most productive panel
most_productive_panel_id = panel_daily_energy.sum(axis=1).idxmax()
energy_per_day = panel_daily_energy.loc[most_productive_panel_id]
plt.figure(figsize=(14,4))
plt.plot(energy_per_day.values)
plt.title(f"Daily energy for most productive panel (ID {most_productive_panel_id})")
plt.xlabel("Day")
plt.ylabel("Energy generated (Wh)")
plt.grid(True)
plt.show()

## Battery

In [ ]:
def battery_required_energy(Tbat, Pn, nc):
    """
    Calculate the required battery energy capacity.
    Parameters:
    - Tbat: Autonomy time (hours)
    - Pn: Mean nominal power (W)
    - nc: Battery efficiency (decimal)
    """
    assert 0 < nc <= 1, "Battery efficiency must be between 0 and 1"
    return (Tbat * Pn) / nc


def battery_autonomy(Qbat, Pn, nc):
    """
    Calculate the battery autonomy time.
    Parameters:
    - Ebat: Battery energy capacity (Wh)
    - Pn: Mean nominal power (W)
    - nc: Battery efficiency (decimal)
    """
    assert 0 < nc <= 1, "Battery efficiency must be between 0 and 1"
    return (Qbat * nc) / Pn

In [ ]:
# Example battery

Qbat = 10  # capacity (Ah)
Vbat = 12  # voltage (V)
Iload = 0.5   # mean load current (A)
nc = 0.9  # battery efficiency (decimal)

Tbat = battery_autonomy(Qbat, Iload * Vbat, nc)
print(f"Battery autonomy: {Tbat:.2f} hours")